In [ ]:
# Code written by Claude (Anthropic, Claude Opus 5.5) for Google Colab compatibility.
# Adapted from the install cell of demo_2/notebooks/fr/07_llm_and_rag_FP16.ipynb (IFT 6758).
# OPTIONAL: once per new Colab runtime (version 2025.07 = Python 3.11, T4 GPU).
INSTALL_COLAB = False  # Set to True to install, then back to False

import sys
import subprocess
from pathlib import Path
from importlib.metadata import version

REPO_URL = "https://github.com/Mahdi267/NHL-data-analysis.git"
REPO_DIR = Path("/content/NHL-data-analysis")
REQ_FILE = Path("/content/requirements_colab.txt")

# Detect Colab; otherwise the local .venv is used
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: install skipped (uv sync --extra llm in the terminal).")
elif not INSTALL_COLAB:
    print("Install skipped. Set INSTALL_COLAB = True on a new Colab runtime.")
else:
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Python 3.11 required: Runtime → Change runtime type → version 2025.07.")

    # 1. Project code (public repository: no credentials)
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull"])

    # 2. Same versions as local (uv.lock), except torch: we keep Colab's,
    #    built for its CUDA driver (the torch in the lock is a CUDA 13 build)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    export = subprocess.check_output(
        [sys.executable, "-m", "uv", "export", "--frozen", "--extra", "llm",
         "--no-dev", "--no-hashes", "--no-emit-project"],
        cwd=REPO_DIR, text=True,
    )
    skip = ("torch==", "triton==", "nvidia-")
    lines = [line for line in export.splitlines() if not line.startswith(skip)]
    lines.append(f"torch=={version('torch').split('+')[0]}")  # Pin the torch already installed
    REQ_FILE.write_text("\n".join(lines))

    # 3. Install into the notebook's Python, then the project in editable mode
    uv_install = [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable]
    subprocess.check_call(uv_install + ["-r", str(REQ_FILE)])
    subprocess.check_call(uv_install + ["--no-deps", "-e", str(REPO_DIR)])
    print("Install done. Runtime → Restart session, then set INSTALL_COLAB = False.")

In [ ]:
# Code written by Claude (Anthropic, Claude Opus 5.5) for Google Colab compatibility.
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Adjust to THIS notebook's folder (notebooks/ or visualizations/): relative paths depend on it
NOTEBOOK_DIR = Path("/content/NHL-data-analysis/ift6758/notebooks")
if IN_COLAB:
    os.chdir(NOTEBOOK_DIR)
print("Working directory:", Path.cwd())

from ift6758.llm.qwen_utils import check_env
check_env()  # Should show Python 3.11 and CUDA GPU: Tesla T4

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path
import pandas as pd
import requests
from ift6758.llm.get_better import (
    extract_text,
    load_function,
    format_test_report,
    all_passed,
)
from ift6758.llm.chat import confirm
from ift6758.llm.prompts import PROFILES
from ift6758.llm.qwen_log import create_log, log_call
from ift6758.llm.qwen_utils import (
    load_qwen,
    ask_qwen,
    check_env,
)
from ift6758.llm.rag import (
    load_doc,
    extract_section,
    split_doc,
    load_embedder,
    embed,
    retrieve,
    format_context,
    DOCS_DIR,
)
from ift6758.llm.test_tidy_llm import run_tidy_tests, compare_to_reference, add_goalie_stops

check_env()  # Demo 2, for reproductibility

In [ ]:
# Code adapted from demo 2
GAME_ID = 2022030411
raw_path = Path(DOCS_DIR.parent / "raw") / f"{GAME_ID}.json"
print("Cached game:", raw_path.resolve())
if not raw_path.exists():
    response = requests.get(
        f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30
    )
    response.raise_for_status()
    downloaded = response.json()
    assert downloaded["id"] == GAME_ID and isinstance(downloaded["plays"], list)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(response.text, encoding="utf-8")
game = json.loads(
    raw_path.read_text(encoding="utf-8")
)  # Converted from JSON to a Python object
assert game["id"] == GAME_ID and isinstance(game["plays"], list)
print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

In [ ]:
# Reference DataFrame: Mahdi's json2tidy, checked by hand against NHL.com (except x and y, not available)
reference = pd.read_csv(DOCS_DIR / "reference_tidy.csv")
REF_GAMES = {
    2022030411: DOCS_DIR.parent / "raw" / "2022030411.json",
    2020020007: Path("data/raw/2020/2020020007.json"),  # Relative to ift6758/notebooks/ (ignored by git)
    2020020020: Path("data/raw/2020/2020020020.json"),
}
# Same download as the cell above, for the games that are not cached
for gid, path in REF_GAMES.items():
    if not Path(path).exists():
        response = requests.get(
            f"https://api-web.nhle.com/v1/gamecenter/{gid}/play-by-play", timeout=30
        )
        response.raise_for_status()
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        Path(path).write_text(response.text, encoding="utf-8")
print(reference.groupby("game_id")["event_type"].value_counts().unstack())

## Q3 plan: one generation per level of context

Same prompt (`tidy_task`, only the assignment + the interface) for every variant, `do_sample=False`.
A single generation per variant, to isolate the effect of the context (no refinement loop:
generic tests do not measure correctness, and exact tests would give the answer to the model).

| Variante | Contexte |
|---|---|
| C0 | None |
| C1 | RAG on the API documentation (Zmalski), same query as Q1 |
| C2 – C3 | *Context built with the help of Claude* |
| C2 | JSON sample: first shot and first goal of the game (not chosen) |
| C3 | JSON sample chosen to show the traps (shot, goal at 1451, empty net goal) |

Evaluation: 3 generic tests (`run_tidy_tests`) + comparison with the reference (`compare_to_reference`).

In [ ]:
# 4 levels of context, same prompt (tidy_task) for all. One generation per level.
CONTEXTS = {}

# C0 : no context
CONTEXTS["C0"] = {"label": "no context", "context": None, "retrieved": None}

# C1 : a little context = RAG on the NHL API documentation (same code as Q1, cell 2)
zmalski_file = "zmalski-nhl-api-reference.md"
zmalski_url = "https://raw.githubusercontent.com/Zmalski/NHL-API-Reference/main/README.md"
text = load_doc(zmalski_url, zmalski_file)
section = extract_section(text, "# NHL Web API Documentation", "# NHL Stats API Documentation")
chunks = split_doc(section)
embedder = load_embedder()
vectors = embed(chunks, embedder)
# Same retrieval query as Q1: finds the "Get Play By Play" endpoint (passage 35)
rag_results = retrieve(PROFILES["gen_code"]["retrieval_query"], chunks, vectors, embedder, k=PROFILES["gen_tidy"]["k"])
CONTEXTS["C1"] = {
    "label": "RAG NHL API doc",
    "context": format_context(rag_results, source=zmalski_file),
    "retrieved": rag_results,
}

# ____ DISCLAMER : CONTEXT BEYOND THIS POINT WAS PROPOSED BY CLAUDE TO HELP GET DECENT RESULTS ____

def build_sample_context(game, sample_events, source):
    """Sample of the real game JSON (game level, chosen plays, their players), in the format of retrieve().
    Same code as the previous context cell, put in a function to build C2 and C3.

    :param game: Game JSON
    :type game: dict
    :param sample_events: eventId of the plays to show
    :type sample_events: tuple
    :param source: Source label for format_context
    :type source: str
    :return: context, passages
    :rtype: tuple[str, list[dict]]
    """
    # Match: just as peek_game_level
    game_level = {
        "id": game["id"],
        "season": game["season"],
        "gameType": game["gameType"],
        "homeTeam": {"id": game["homeTeam"]["id"], "abbrev": game["homeTeam"]["abbrev"]},
        "awayTeam": {"id": game["awayTeam"]["id"], "abbrev": game["awayTeam"]["abbrev"]},
    }
    # Chosen plays and their position in game["plays"]
    sample_plays = [(i, p) for i, p in enumerate(game["plays"]) if p["eventId"] in sample_events]
    # peek_shooter_keys ("PlayerId" in k) + goalie
    player_ids = {
        v for _, p in sample_plays for k, v in p["details"].items()
        if "PlayerId" in k or k == "goalieInNetId"
    }
    # rosterSpots: same search as peek_player_name, without useless fields
    roster = [
        {key: r[key] for key in ("playerId", "teamId", "firstName", "lastName")}
        for r in game["rosterSpots"] if r["playerId"] in player_ids
    ]
    # Same format as retrieve() to use format_context()
    passages = [("game level", game_level)]
    passages += [(f"plays[{i}]", p) for i, p in sample_plays]
    passages += [("rosterSpots (players of the plays above)", roster)]
    passages = [
        {"rank": rank, "index": label, "score": None, "text": json.dumps(part, indent=2)}
        for rank, (label, part) in enumerate(passages, start=1)
    ]
    context = (
        "Excerpts of one real NHL play-by-play game JSON. All games have the same structure.\n\n"
        + format_context(passages, source=source)
    )
    return context, passages

# C2 : medium context = first shot on goal and first goal of the game (not chosen by us)
first_shot = next(p["eventId"] for p in game["plays"] if p["typeDescKey"] == "shot-on-goal")
first_goal = next(p["eventId"] for p in game["plays"] if p["typeDescKey"] == "goal")
c2_context, c2_passages = build_sample_context(game, (first_shot, first_goal), f"{GAME_ID}.json")
CONTEXTS["C2"] = {"label": f"JSON sample, events {first_shot}/{first_goal}", "context": c2_context, "retrieved": c2_passages}

# C3 : a lot of context = plays chosen to show every trap (shot, goal at 1451, empty net goal)
c3_context, c3_passages = build_sample_context(game, (55, 87, 1001), f"{GAME_ID}.json")
CONTEXTS["C3"] = {"label": "JSON sample, events 55/87/1001", "context": c3_context, "retrieved": c3_passages}

for name, c in CONTEXTS.items():
    print(name, "|", c["label"], "|", len(c["context"] or ""), "characters")

In [ ]:
# I wrote this prompt reusing the structure proposed by Claude and by the article, but with only the information
# that I knew myself: the wanted format (like Mahdi's) and the information asked by the assignment.
# - The JSON is the play-by-play of one game from the NHL web API:
#  https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play
tidy_task = """
Write a Python function json2tidy_llm(raw_data).
Goal: convert all the events of one NHL game into a tidy pandas DataFrame.

Input:
- raw_data is either a str (path to a JSON file, read as UTF-8) or a dict (already loaded).

Task:
- Keep only shots on goal and goals. Ignore missed shots and blocked shots.
- One row per kept event, uniquely identified by the game ID and the event ID.
- For each event include at least: game ID, event ID, season, game type, period, time in the period,
  the team that took the shot, whether it is a shot or a goal, the coordinates on the ice,
  the shooter name, the goalie name, the shot type, whether the net was empty,
  and, for goals, whether the goal was scored at even strength, shorthanded or on the power play.
- Keep unavailable values as missing. Never replace them with 0 or False.

Output interface (used by automated tests):
- Columns: game_id, event_id, season, game_type, period, time, team_id, event_type,
  x_coord, y_coord, shooter, goalie, shot_type, empty_net, strength.
- event_type is "shot-on-goal" or "goal". game_type is the numeric game type of the game.
- shooter and goalie are full names: "First Last".
- empty_net is True or False.
- strength is "even", "shorthanded" or "power play" for goals, and missing for shots.

Use pandas and json. Include imports and a docstring.
Return only json2tidy_llm and its imports in a Python code block.
"""

In [ ]:
model, tokenizer, device = load_qwen()
log_path = create_log()
print("Log :", log_path)

# Code suggested by Claude to embelish presentation
def banner(title):
    print(f"\n{'=' * 15} {title} {'=' * 15}")

In [ ]:
GENERATED_DIR = DOCS_DIR.parent.parent / "llm" / "generated"  # Generated code of each variant, for the report
TASK_VERSION = "tidy_naive_v0"


def run_variant(name, task, ctx):
    """One generation with one context, then extraction, tests and comparison with the reference.
    Same steps as the Q1 loop, without the refinement.

    :param name: Variant name (C0, C1...)
    :type name: str
    :param task: Task prompt
    :type task: str
    :param ctx: Entry of CONTEXTS
    :type ctx: dict
    :return: One row of the summary table
    :rtype: dict
    """
    banner(f"{name} : code generation ({ctx['label']})")
    gen_result = ask_qwen(
        model=model,
        tokenizer=tokenizer,
        log_path=log_path,
        prompt=task,
        context=ctx["context"],
        profile=PROFILES["gen_tidy"],
        extra={
            "variant": name,
            "context_label": ctx["label"],
            "task_version": TASK_VERSION,
            "retrieved": ctx["retrieved"],
        },
    )
    row = {
        "variant": name,
        "context": ctx["label"],
        "n_token_in": gen_result["n_token_in"],
        "n_token_out": gen_result["n_token_out"],
        "duration_s": round(gen_result["duration_s"], 1),
        "truncated": gen_result["n_token_out"] >= PROFILES["gen_tidy"]["max_new_tokens"],
    }

    # Extraction and human in the loop validation
    try:
        tidy_code = extract_text(gen_result["answer"], tag="code")
    except ValueError as e:
        tidy_code = gen_result["answer"]  # no code generated
        test_results = {"0. Code extraction": f"ERROR : {e}"}
    else:
        GENERATED_DIR.mkdir(parents=True, exist_ok=True)
        (GENERATED_DIR / f"tidy_{name}.py").write_text(tidy_code, encoding="utf-8")
        if not confirm("Execute and test code ?"):
            row["tests"] = "not executed"
            return row
        try:
            fn = load_function(tidy_code, "json2tidy_llm")
        except Exception as e:  # SyntaxError, missing import...
            test_results = {"0. Code loading": f"ERROR : {type(e).__name__}: {e}"}
        else:
            test_results = run_tidy_tests(fn)
            # Comparison with the reference (can crash if a column is missing)
            try:
                comparison = compare_to_reference(fn, reference, REF_GAMES)
            except Exception as e:
                row["comparison"] = f"ERROR : {type(e).__name__}: {e}"
            else:
                row.update(comparison["rows"])
                row.update({f"{col} %": pct for col, pct in comparison["agreement_%"].items()})
                row["mismatch_examples"] = comparison["mismatch_examples"]

    report = format_test_report(test_results)
    banner(f"{name} : test results")
    print(report)
    row["tests_passed"] = f"{sum(s == 'OK' for s in test_results.values())}/{len(test_results)}"
    row["all_passed"] = all_passed(test_results)
    row["report"] = report
    return row

In [ ]:
summary = []  # One row per variant, seen at the end
for name in ["C0", "C1", "C2", "C3"]:  # Long ....
    summary.append(run_variant(name, tidy_task, CONTEXTS[name]))

In [ ]:
summary_df = pd.DataFrame(summary).set_index("variant")
summary_df.drop(columns=["report", "mismatch_examples"], errors="ignore").T

In [ ]:
logs = pd.read_json(log_path, lines=True)
logs[["variant", "context_label", "task_version", "n_token_in", "n_token_out", "duration_s"]]

New section for the caracteristics implementation

In [ ]:
stops_task = """
Write a Python function add_goalie_stops_llm(df).
Goal: add a feature "goalie_stops" to a tidy DataFrame of NHL shots and goals.

Input:
- df is a pandas DataFrame with one row per shot on goal or goal, already in chronological order
  inside each game. Columns include: game_id, event_id, event_type, goalie.
- event_type is "shot-on-goal" or "goal". goalie is the goalie's full name, missing when the net is empty.
- The context shows the first rows of a real DataFrame.

Rule for goalie_stops (number of shots stopped by the goalie so far in the game, current event included):
- If event_type is "shot-on-goal", add 1 to the counter of the corresponding goalie.
- If it is the first time in a game that a goalie stops a shot, the counter starts at 1, or at 0 if it is a goal.
- If there are previous events with that goalie in the same game, take the value from the last event
  and add 1 if the current event_type is "shot-on-goal".
- A game can have more than one goalie per team: keep one counter per goalie and per game.

Requirements:
- Do not sort the rows. event_id is not in chronological order.
- If goalie is missing (empty net), goalie_stops is missing. Never replace it with 0.
- Return a copy of df with the new column goalie_stops. Same rows, same order. Do not modify df.
- Use pandas. Include imports and a docstring.
Return only add_goalie_stops_llm and its imports in a Python code block.
"""

In [ ]:
stops_context = (
    "First rows of a real tidy DataFrame (game 2022030411), in chronological order:\n\n"
    + reference[reference["game_id"] == 2022030411]
    .head(11)[["game_id", "event_id", "period", "time", "event_type", "goalie", "empty_net"]]
    .to_string(index=False)
)

In [ ]:
# Qwen writes add_goalie_stops_llm, we run it and the manual version on the same input, and log both
gen_result = ask_qwen(
    model=model,
    tokenizer=tokenizer,
    log_path=log_path,
    prompt=stops_task,
    context=stops_context,
    profile=PROFILES["gen_tidy"],
    extra={"task_version": "stops_v0", "context_label": "tidy DataFrame sample"},
)
stops_code = extract_text(gen_result["answer"], tag="code")
llm_stops = load_function(stops_code, "add_goalie_stops_llm")  # Qwen's version, the manual one is kept

# Same input for both versions
llm_out = llm_stops(reference)
manual_out = add_goalie_stops(reference)

# Final count per goalie (last value of the counter), side by side
totals = pd.DataFrame({
    "manual": manual_out.dropna(subset=["goalie"]).groupby(["game_id", "goalie"])["goalie_stops"].last(),
    "qwen": llm_out.dropna(subset=["goalie"]).groupby(["game_id", "goalie"])["goalie_stops"].last(),
})
manual_stops = manual_out["goalie_stops"].astype("Float64")
llm_stops_col = pd.to_numeric(llm_out["goalie_stops"]).astype("Float64")
# NaN == NaN counts as identical (empty net rows)
same_rows = ((manual_stops == llm_stops_col).fillna(False) | (manual_stops.isna() & llm_stops_col.isna())).mean()

# Second log record: the manual version and the comparison
log_call(
    log_path,
    messages=[],
    result={"answer": "manual add_goalie_stops (test_tidy_llm.py)", "n_token_out": 0, "duration_s": 0},
    extra={
        "task_version": "stops_v0",
        "qwen_code": stops_code,
        "rows_identical_%": round(100 * same_rows, 1),
        "totals": totals.reset_index().astype(str).to_dict("records"),
    },
)
print(f"Rows identical: {100 * same_rows:.1f} %")
totals